# 03 Hybrid

Hybrid pipeline: RL proposal -> feasibility layer -> GLS refinement.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve() / 'src'))

from dvrptw_bench.data.instance_filters import find_rc_instances
from dvrptw_bench.data.solomon_parser import parse_solomon
from dvrptw_bench.dynamic.simulator import DynamicSimulator
from dvrptw_bench.hybrid.hybrid_runner import run_hybrid
from dvrptw_bench.hybrid.feasibility_layer import FeasibilityLayer
from dvrptw_bench.common.typing import Route, Solution
from dvrptw_bench.viz.route_plot import plot_routes
from dvrptw_bench.rl.policies import build_policy
from dvrptw_bench.viz.dashboard import render_dashboard

In [2]:
dataset_root = Path('../dataset')
instances = find_rc_instances(dataset_root)
print(instances)
print(f"Found {len(instances)} RC instances.")
assert instances, 'Place RC*.txt files under ../dataset/solomon_rc100/'
instance = parse_solomon(instances[0])
instance.instance_id

[PosixPath('../dataset/solomon_rc100/RC2_10_9.TXT')]
Found 1 RC instances.


'RC2_10_9.TXT'

## Static hybrid run and comparison plot

In [ ]:
rl4coPolicy = build_policy('rl4co')
rl4coPolicy.train()

In [ ]:
hybrid_sol, timings = run_hybrid(instance, rl4coPolicy, budget_s=2, gls_time_share=0.5, feasibility_mode='repair')
hybrid_sol.total_distance, timings

Refining with ORTools for up to 1.0s...


(47453.82369768667,
 {'inference_s': 0.0001355829881504178,
  'local_search_s': 0.9999322085059248,
  'total_s': 255.07914783299202})

In [5]:
from dvrptw_bench.heuristics.ortools_solver import ORToolsVRPTWSolver


ORToolsVRPTWSolver().solve(instance, 1).total_distance

47412.22905921087

In [4]:
out = Path('../outputs/notebook_hybrid')
out.mkdir(parents=True, exist_ok=True)
_ = plot_routes(instance, hybrid_sol, out / 'hybrid_route.png')
print('Saved:', out / 'hybrid_route.png')

Saved: ../outputs/notebook_hybrid/hybrid_route.png


## Dynamic hybrid run (epsilon=0.5)

In [5]:
sim = DynamicSimulator(instance)
dyn_sol, events, scenario = sim.run(
    lambda snap_inst, budget: run_hybrid(snap_inst, policy_name='ga', budget_s=budget, gls_time_share=0.9, feasibility_mode='repair')[0],
    epsilon=0.5, budget_s=10, seed=1
)
len(events), None if dyn_sol is None else dyn_sol.total_distance

(0, None)

## Feasibility layer behavior (forced infeasible proposal)

In [6]:
bad = Solution(strategy='forced_bad', routes=[Route(vehicle_id=0, node_ids=[c.id for c in instance.customers] * 2)])
layer = FeasibilityLayer(mode='repair')
fixed = layer.enforce(bad, instance)
fixed.strategy, len(fixed.routes)

('pmca', 25)

## CLI + dashboard usage examples

In [7]:
import subprocess
cmd = [
    sys.executable, '-m', 'dvrptw_bench.cli.hybrid_cli',
    'eval-hybrid', '--policy', 'ga', '--instance', instance.instance_id,
    '--epsilon', '0.5', '--budget', '10', '--dataset-root', '../dataset', '--output-root', '../outputs'
]
print(' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

/Users/giuseppe/Documents/personal/fyp-vrp/fyp-vrp-venv/bin/python -m dvrptw_bench.cli.hybrid_cli eval-hybrid --policy ga --instance RC201.txt --epsilon 0.5 --budget 10 --dataset-root ../dataset --output-root ../outputs

/Users/giuseppe/Documents/personal/fyp-vrp/fyp-vrp-venv/bin/python: Error while finding module specification for 'dvrptw_bench.cli.hybrid_cli' (ModuleNotFoundError: No module named 'dvrptw_bench')



In [8]:
# If you have a run folder with summary.csv, render dashboard plots:
# render_dashboard(Path('../outputs/<run_id>'))